# Manual Inspection of Error Analysis Summary

In [2]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
INPUT_PATH = PROJECT_ROOT / "results" / "error_analysis_summary" / "manual_inspection.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "error_analysis_summary_final"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH)

df.shape

(60, 15)

### Manual categ counts

In [3]:
manual_category_counts = (
    df["manual_category"]
    .value_counts()
    .reset_index()
)

manual_category_counts.columns = ["manual_category", "count"]

manual_category_counts["percentage"] = (
    manual_category_counts["count"] / len(df) * 100
).round(1)

manual_category_counts.to_csv(
    OUTPUT_DIR / "manual_category_counts.csv",
    index=False,
)

manual_category_counts

,manual_category,count,percentage
0,span_segmentation_issue,18,30.0
1,noise_or_formatting,13,21.7
2,missed_gold_span,12,20.0
3,boundary_issue_only,6,10.0
4,anecdote_assumption_ambiguity,5,8.3
5,testimony_assumption_ambiguity,3,5.0
6,testimony_anecdote_ambiguity,1,1.7
7,other_assumption_ambiguity,1,1.7
8,testimony_statistics_ambiguity,1,1.7


### Manual category by automatic error type

In [4]:
manual_by_error_type = pd.crosstab(
    df["error_type"],
    df["manual_category"]
)

manual_by_error_type.to_csv(
    OUTPUT_DIR / "manual_category_by_error_type.csv"
)

manual_by_error_type

manual_category,anecdote_assumption_ambiguity,boundary_issue_only,missed_gold_span,noise_or_formatting,other_assumption_ambiguity,span_segmentation_issue,testimony_anecdote_ambiguity,testimony_assumption_ambiguity,testimony_statistics_ambiguity
error_type,,,,,,,,,
overlapping_different_label,5,0,0,1,1,3,1,1,0
same_label_boundary_difference,0,6,0,2,0,4,0,0,0
same_label_segmentation_difference,0,0,0,1,0,11,0,0,0
unmatched_gold_span,0,0,12,0,0,0,0,0,0
unmatched_prediction,0,0,0,9,0,0,0,2,1


A cross-tabulation of automatic error types and manually assigned categories shows that the automatic categories were useful for retrieving different kinds of disagreement, but did not map one-to-one onto substantive error types. All unmatched_gold_span cases were manually coded as missed gold spans, and most same_label_segmentation_difference cases were coded as span segmentation issues. However, other automatic categories were more heterogeneous. For example, same_label_boundary_difference included minor boundary issues, more substantial segmentation problems, and noise or formatting artefacts. Similarly, overlapping_different_label mostly captured label ambiguities, especially anecdote/assumption disagreements, but also included cases where the main issue was incomplete span selection. This confirms the need for manual qualitative inspection rather than relying only on automatically assigned error types.

In [5]:
gold_by_predicted_label = pd.crosstab(
    df["gold_label"].fillna("none").replace("", "none"),
    df["pred_label"].fillna("none").replace("", "none"),
)

gold_by_predicted_label.to_csv(
    OUTPUT_DIR / "manual_error_gold_by_predicted_label.csv"
)

gold_by_predicted_label

pred_label,anecdote,assumption,none,other,statistics,testimony
gold_label,,,,,,
anecdote,4,4,3,0,0,0
assumption,2,3,3,0,0,0
none,5,4,0,2,1,0
other,0,2,2,0,0,0
statistics,0,0,0,0,1,0
testimony,3,1,4,0,0,16


The gold-by-predicted label table shows that many inspected cases were not straightforward label confusions. The largest cell was testimony predicted as testimony, indicating that many testimony-related disagreements concerned boundary, segmentation, or formatting rather than the evidence label itself. Label-level confusions were nevertheless visible, especially anecdote/assumption disagreements. The none cases represent missing or unmatched spans: gold labels paired with none indicate missed gold spans, while none paired with a predicted label indicates unmatched model predictions.